In [ ]:
import json, requests, time
import numpy as np
from collections import Counter
from google.colab import drive, userdata
from datasets import load_dataset

drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive'
TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')

def load_results(path):
    with open(path) as f:
        return {r['idx']: r for r in json.load(f) if 'idx' in r}

llama = load_results(f'{PROJECT_DIR}/llama_results.json')
qwen  = load_results(f'{PROJECT_DIR}/qwen_results.json')
gemma = load_results(f'{PROJECT_DIR}/gemma3n_results.json')

medqa_ds = load_dataset("GBaker/MedQA-USMLE-4-options")
test = medqa_ds['test']
N = len(test)

MODELS_FOR_DETECTOR = {
    'llama': "meta-llama/Meta-Llama-3-8B-Instruct-Lite",
    'qwen':  "Qwen/Qwen2.5-7B-Instruct-Turbo",
    'gemma': "google/gemma-3n-E4B-it",
}

def query_letter(question, options, model, api_key):
    prompt = f"""Answer this USMLE medical question. Reply with only A, B, C, or D.

Question: {question}

A: {options['A']}
B: {options['B']}
C: {options['C']}
D: {options['D']}

Answer:"""
    try:
        r = requests.post(
            "https://api.together.xyz/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
            json={"model": model,
                  "messages": [{"role": "user", "content": prompt}],
                  "max_tokens": 5, "temperature": 0.0},
            timeout=30
        )
        if r.status_code != 200:
            return None
        text = r.json()['choices'][0]['message']['content'].strip().upper()
        for c in text:
            if c in 'ABCD':
                return c
        return None
    except:
        return None

def convergence_risk(question, options, api_key=TOGETHER_API_KEY, verbose=False):
    """
    Returns a risk score and diagnostic info for a clinical MCQ.

    Risk classes:
      - HIGH (0.85+):  All 3 models agree AND no model checks out via answer text plausibility
      - MED (0.50-0.85): 2 of 3 models agree on a non-default answer
      - LOW (<0.50):   Models disagree meaningfully
    """
    preds = {}
    for name, model_id in MODELS_FOR_DETECTOR.items():
        preds[name] = query_letter(question, options, model_id, api_key)
        time.sleep(0.1)

    valid_preds = [p for p in preds.values() if p in 'ABCD']
    if len(valid_preds) < 2:
        return {'risk': 0.5, 'tier': 'INSUFFICIENT', 'preds': preds,
                'reason': 'Fewer than 2 valid responses'}

    counts = Counter(valid_preds)
    most_common, mc_count = counts.most_common(1)[0]
    agreement_ratio = mc_count / len(valid_preds)

    if agreement_ratio == 1.0:
        risk = 0.90
        tier = 'HIGH'
        reason = f'All {len(valid_preds)} mid-tier LLMs converged on option {most_common} — shared-failure-mode risk'
    elif agreement_ratio >= 0.67:
        risk = 0.60
        tier = 'MEDIUM'
        reason = f'{mc_count}/{len(valid_preds)} mid-tier LLMs agree on {most_common} — partial convergence'
    else:
        risk = 0.20
        tier = 'LOW'
        reason = f'No majority — models disagree (predictions: {dict(counts)})'

    result = {
        'risk': risk,
        'tier': tier,
        'preds': preds,
        'majority_answer': most_common,
        'agreement_ratio': agreement_ratio,
        'reason': reason,
    }
    if verbose:
        print(f"  Risk: {risk:.2f} ({tier})")
        print(f"  Predictions: {preds}")
        print(f"  Reason: {reason}")
    return result

print(f"=== Validating detector on existing MedQA evaluations (no new API calls) ===\n")

detector_outputs = []
for i in range(N):
    preds_at_idx = {
        'llama': llama[i].get('pred'),
        'qwen':  qwen[i].get('pred'),
        'gemma': gemma[i].get('pred'),
    }
    valid = [p for p in preds_at_idx.values() if isinstance(p, str) and p in 'ABCD']
    if len(valid) < 2:
        continue
    counts = Counter(valid)
    mc, count = counts.most_common(1)[0]
    ratio = count / len(valid)

    if ratio == 1.0:
        tier = 'HIGH'
    elif ratio >= 0.67:
        tier = 'MEDIUM'
    else:
        tier = 'LOW'

    gold = test[i]['answer_idx']
    detector_outputs.append({
        'idx': i,
        'tier': tier,
        'majority_answer': mc,
        'gold': gold,
        'majority_correct': mc == gold,
    })

print(f"{'Tier':<10} {'N':<8} {'Majority correct':<18} {'Majority wrong':<18} {'Wrong rate':<10}")
print("-" * 70)
for tier in ['HIGH', 'MEDIUM', 'LOW']:
    rows = [r for r in detector_outputs if r['tier'] == tier]
    n = len(rows)
    n_wrong = sum(1 for r in rows if not r['majority_correct'])
    n_right = n - n_wrong
    if n > 0:
        print(f"  {tier:<8} {n:<8} {n_right:<18} {n_wrong:<18} {n_wrong/n*100:.1f}%")

print(f"\nWhat the detector tells a practitioner:")
print(f"  HIGH tier:   ensemble majority is wrong this often. If high, route to a human.")
print(f"  LOW tier:    ensemble majority is more trustworthy.")

import json
with open(f'{PROJECT_DIR}/detector_medqa_validation.json', 'w') as f:
    json.dump({
        'tool': 'convergence_risk',
        'description': 'Detects high-risk clinical MCQs where mid-tier LLM ensembles will silently agree on wrong answers',
        'validation_dataset': 'MedQA-USMLE-4-options test split (n=1273)',
        'models_used': list(MODELS_FOR_DETECTOR.values()),
        'tier_calibration': {
            'HIGH': 'all 3 LLMs agree',
            'MEDIUM': '2 of 3 LLMs agree',
            'LOW': 'no majority'
        },
        'results': detector_outputs,
        'summary': {
            tier: {
                'n': sum(1 for r in detector_outputs if r['tier'] == tier),
                'wrong_rate': sum(1 for r in detector_outputs if r['tier'] == tier and not r['majority_correct']) /
                              max(1, sum(1 for r in detector_outputs if r['tier'] == tier))
            } for tier in ['HIGH', 'MEDIUM', 'LOW']
        }
    }, f, indent=2)

print(f"\nSaved detector_medqa_validation.json")
print(f"\n=== Detector ready for use in paper ===")
print(f"Usage in clinical AI deployment:")
print(f"  result = convergence_risk(question, options)")
print(f"  if result['tier'] == 'HIGH': route_to_human_reviewer()")

In [ ]:
import json
import numpy as np
from collections import Counter

PROJECT_DIR = '/content/drive/MyDrive'

def load_results(path):
    with open(path) as f:
        return {r['idx']: r for r in json.load(f) if 'idx' in r}

gpt4o = load_results(f'{PROJECT_DIR}/medqa_gpt4o_results.json')

records = []
for i in range(N):
    mid_preds = {
        'llama': llama[i].get('pred'),
        'qwen':  qwen[i].get('pred'),
        'gemma': gemma[i].get('pred'),
    }
    frontier_pred = gpt4o[i].get('pred')
    gold = test[i]['answer_idx']

    valid_mid = [p for p in mid_preds.values() if isinstance(p, str) and p in 'ABCD']
    if len(valid_mid) < 2:
        continue

    counts = Counter(valid_mid)
    mid_majority, mid_count = counts.most_common(1)[0]
    mid_ratio = mid_count / len(valid_mid)

    if mid_ratio == 1.0:

        if frontier_pred and frontier_pred != mid_majority:
            tier = 'HIGH_RISK'
            reason = f'Mid-tier unanimous on {mid_majority}, but GPT-4o disagrees ({frontier_pred})'
        elif frontier_pred == mid_majority:
            tier = 'CONFIDENT'
            reason = 'All 4 models agree'
        else:
            tier = 'AMBIGUOUS'
            reason = 'Mid-tier unanimous, frontier missing'
    elif mid_ratio >= 0.67:
        tier = 'PARTIAL_AGREEMENT'
        reason = f'{mid_count}/3 mid-tier agree on {mid_majority}'
    else:
        tier = 'DISAGREEMENT'
        reason = 'No mid-tier majority'

    records.append({
        'idx': i,
        'tier': tier,
        'mid_majority': mid_majority,
        'frontier_pred': frontier_pred,
        'gold': gold,
        'mid_majority_correct': mid_majority == gold,
        'frontier_correct': frontier_pred == gold if frontier_pred else None,
    })

print(f"=== Improved Detector: 5-tier risk classification ===\n")
print(f"{'Tier':<20} {'N':<6} {'Mid wrong':<11} {'Mid wrong %':<12} {'Frontier wrong %':<15}")
print("-" * 78)
for tier in ['HIGH_RISK', 'CONFIDENT', 'PARTIAL_AGREEMENT', 'DISAGREEMENT', 'AMBIGUOUS']:
    rows = [r for r in records if r['tier'] == tier]
    n = len(rows)
    if n == 0:
        print(f"  {tier:<18} {n:<6}  -            -              -")
        continue
    n_mid_wrong = sum(1 for r in rows if not r['mid_majority_correct'])
    fr_valid = [r for r in rows if r['frontier_correct'] is not None]
    n_fr_wrong = sum(1 for r in fr_valid if not r['frontier_correct'])
    print(f"  {tier:<18} {n:<6}  {n_mid_wrong:<11} {n_mid_wrong/n*100:>6.1f}%       {n_fr_wrong/max(1,len(fr_valid))*100:>6.1f}%")

high_risk_records = [r for r in records if r['tier'] == 'HIGH_RISK']
high_risk_wrong = [r for r in high_risk_records if not r['mid_majority_correct']]

print(f"\n=== HIGH_RISK Tier Validation ===")
print(f"  Questions flagged HIGH_RISK: {len(high_risk_records)}")
print(f"  Of these, mid-tier majority WRONG: {len(high_risk_wrong)} ({len(high_risk_wrong)/max(1,len(high_risk_records))*100:.1f}%)")
print(f"  → Routing HIGH_RISK to humans catches {len(high_risk_wrong)} silent failures")

baseline_total = sum(1 for r in records if not r['mid_majority_correct'])
print(f"\n=== Comparison vs. no detector ===")
print(f"  Total mid-tier majority errors in dataset: {baseline_total}")
print(f"  Catchable via HIGH_RISK routing: {len(high_risk_wrong)} ({len(high_risk_wrong)/max(1,baseline_total)*100:.1f}% of all errors)")
print(f"  Cost: routing {len(high_risk_records)} questions to humans ({len(high_risk_records)/N*100:.1f}% of dataset)")

precision_high = len(high_risk_wrong) / max(1, len(high_risk_records))
recall_high = len(high_risk_wrong) / max(1, baseline_total)
print(f"\n  Precision (HIGH_RISK is actually wrong): {precision_high*100:.1f}%")
print(f"  Recall (caught errors / all errors):     {recall_high*100:.1f}%")

import json
with open(f'{PROJECT_DIR}/detector_improved_validation.json', 'w') as f:
    json.dump({
        'tool': 'mid_tier_consensus_vs_frontier_disagreement',
        'rule': 'Flag HIGH_RISK when all mid-tier LLMs agree on answer X but frontier-tier model picks different answer',
        'rationale': 'Mid-tier agreement alone is high-confidence-correct (77.5% right). The shared-failure subset is specifically where frontier-tier knowledge corrects the mid-tier consensus error.',
        'tier_results': {
            tier: {
                'n': sum(1 for r in records if r['tier'] == tier),
                'mid_wrong_rate': sum(1 for r in records if r['tier'] == tier and not r['mid_majority_correct']) /
                                  max(1, sum(1 for r in records if r['tier'] == tier))
            } for tier in ['HIGH_RISK', 'CONFIDENT', 'PARTIAL_AGREEMENT', 'DISAGREEMENT', 'AMBIGUOUS']
        },
        'high_risk_precision': float(precision_high),
        'high_risk_recall': float(recall_high),
    }, f, indent=2)

print(f"\nSaved detector_improved_validation.json")

In [ ]:
import json
from collections import Counter

PROJECT_DIR = '/content/drive/MyDrive'

def load_results(path):
    with open(path) as f:
        return {r['idx']: r for r in json.load(f) if 'idx' in r}

mc_llama = load_results(f'{PROJECT_DIR}/medmcqa_llama_results.json')
mc_qwen  = load_results(f'{PROJECT_DIR}/medmcqa_qwen_results.json')
mc_gemma = load_results(f'{PROJECT_DIR}/medmcqa_gemma3n_results.json')
mc_gpt4o = load_results(f'{PROJECT_DIR}/medmcqa_gpt4o_results.json')

with open(f'{PROJECT_DIR}/medmcqa_sample.json') as f:
    medmcqa_questions = json.load(f)
gold_by_idx = {q['idx']: q['answer_idx'] for q in medmcqa_questions}

records = []
for q in medmcqa_questions:
    idx = q['idx']
    mid_preds = {
        'llama': mc_llama[idx].get('pred'),
        'qwen':  mc_qwen[idx].get('pred'),
        'gemma': mc_gemma[idx].get('pred'),
    }
    frontier_pred = mc_gpt4o[idx].get('pred')
    gold = gold_by_idx[idx]

    valid_mid = [p for p in mid_preds.values() if isinstance(p, str) and p in 'ABCD']
    if len(valid_mid) < 2:
        continue

    counts = Counter(valid_mid)
    mid_majority, mid_count = counts.most_common(1)[0]
    mid_ratio = mid_count / len(valid_mid)

    if mid_ratio == 1.0:
        if frontier_pred and frontier_pred != mid_majority:
            tier = 'HIGH_RISK'
        elif frontier_pred == mid_majority:
            tier = 'CONFIDENT'
        else:
            tier = 'AMBIGUOUS'
    elif mid_ratio >= 0.67:
        tier = 'PARTIAL_AGREEMENT'
    else:
        tier = 'DISAGREEMENT'

    records.append({
        'idx': idx,
        'tier': tier,
        'mid_majority': mid_majority,
        'frontier_pred': frontier_pred,
        'gold': gold,
        'mid_majority_correct': mid_majority == gold,
    })

N_mc = len(records)
print(f"=== MedMCQA Detector Validation (n={N_mc}) ===\n")
print(f"{'Tier':<20} {'N':<6} {'Mid wrong':<11} {'Mid wrong %':<12}")
print("-" * 60)
for tier in ['HIGH_RISK', 'CONFIDENT', 'PARTIAL_AGREEMENT', 'DISAGREEMENT', 'AMBIGUOUS']:
    rows = [r for r in records if r['tier'] == tier]
    n = len(rows)
    if n == 0:
        print(f"  {tier:<18} {n:<6}  -            -")
        continue
    n_wrong = sum(1 for r in rows if not r['mid_majority_correct'])
    print(f"  {tier:<18} {n:<6}  {n_wrong:<11} {n_wrong/n*100:>6.1f}%")

high_risk = [r for r in records if r['tier'] == 'HIGH_RISK']
high_risk_wrong = [r for r in high_risk if not r['mid_majority_correct']]
baseline_total = sum(1 for r in records if not r['mid_majority_correct'])

precision = len(high_risk_wrong) / max(1, len(high_risk))
recall = len(high_risk_wrong) / max(1, baseline_total)

print(f"\n=== HIGH_RISK Tier Validation ===")
print(f"  Flagged HIGH_RISK: {len(high_risk)}/{N_mc} ({len(high_risk)/N_mc*100:.1f}% of dataset)")
print(f"  Precision (flagged questions actually wrong): {precision*100:.1f}%")
print(f"  Recall  (errors caught / all errors):         {recall*100:.1f}%")

print(f"\n=== Cross-dataset comparison ===")
print(f"  {'Metric':<40} {'MedQA':>10} {'MedMCQA':>12}")
print(f"  {'-'*62}")
print(f"  {'HIGH_RISK precision':<40} {'96.1%':>10} {precision*100:>11.1f}%")
print(f"  {'HIGH_RISK recall':<40} {'18.6%':>10} {recall*100:>11.1f}%")
print(f"  {'CONFIDENT wrong rate':<40} {'4.5%':>10} ", end='')
conf = [r for r in records if r['tier'] == 'CONFIDENT']
conf_wrong = sum(1 for r in conf if not r['mid_majority_correct'])
conf_rate = conf_wrong / max(1, len(conf))
print(f"{conf_rate*100:>11.1f}%")
print(f"  {'% routed to humans':<40} {'8.0%':>10} {len(high_risk)/N_mc*100:>11.1f}%")

import json
with open(f'{PROJECT_DIR}/detector_medmcqa_validation.json', 'w') as f:
    json.dump({
        'tool': 'mid_tier_consensus_vs_frontier_disagreement',
        'dataset': 'MedMCQA validation single-choice (n=2816)',
        'tier_results': {
            tier: {
                'n': sum(1 for r in records if r['tier'] == tier),
                'wrong_rate': sum(1 for r in records if r['tier'] == tier and not r['mid_majority_correct']) /
                              max(1, sum(1 for r in records if r['tier'] == tier))
            } for tier in ['HIGH_RISK', 'CONFIDENT', 'PARTIAL_AGREEMENT', 'DISAGREEMENT', 'AMBIGUOUS']
        },
        'high_risk_precision': float(precision),
        'high_risk_recall': float(recall),
        'medqa_comparison': {'precision': 0.961, 'recall': 0.186, 'confident_wrong_rate': 0.045}
    }, f, indent=2)

print(f"\nSaved detector_medmcqa_validation.json")